In [27]:
import numpy as np

In [28]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

---

In [29]:
# Forward pass - Python
def forwardPy(inp, W, bias):
    z = 0
    for i in range(len(inp)):
        z += (inp[i] * W[i])
    z += bias
    a = sigmoid(z)
    return z, a

In [30]:
# Forward pass - Numpy
def forwardNp(inp, W, bias):
    z = np.dot(inp, W) + bias
    a = sigmoid(z)
    return z, a

In [31]:
# Test forward pass
num_feats = 2
W = np.random.randn(num_feats)
bias = np.random.randn()
inp = np.random.randn(num_feats)

assert forwardPy(inp, W, bias) == forwardNp(inp, W, bias)

---

In [32]:
# MSE loss - Python - works only with lists/tuples
def MSELossPy(preds, targs):
    loss = 0
    len_preds = len(preds)
    for i in range(len_preds):
        loss += (preds[i] - targs[i]) ** 2
    loss /= len_preds
    return loss

assert MSELossPy([0, 0], [0, 1]) == 0.5
assert MSELossPy([1, 1], [1, 1]) == 0
assert MSELossPy([0, 0], [1, 1]) == 1

In [33]:
# MSE loss - Numpy - works with numpy arrays, and scalars
def MSELossNp(preds, targs):
    return np.mean((preds - targs) ** 2)

assert MSELossNp(np.array([0, 0]), np.array([0, 1])) == 0.5
assert MSELossNp(np.array([1, 1]), np.array([1, 1])) == 0
assert MSELossNp(np.array([0, 0]), np.array([1, 1])) == 1
assert MSELossNp(0, 0) == 0
assert MSELossNp(0, 1) == 1


---

In [34]:
# XOR dataset
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]]) # (4, 2)
y = np.array([0, 1, 1, 0]) # (4,)
num_samples = len(X) # 4

print(X[0])
print(X[:, 0])
print(X[0, :])

[0 0]
[0 0 1 1]
[0 0]


---

In [35]:
# Model params - One neuron
num_features = X.shape[1]
W = np.random.randn(num_features)
bias = np.random.randn()

print(W)
print(bias)

[-1.00881069  0.84097331]
-0.9640686983792981


In [36]:
z, a = forwardNp(X, W, bias)
loss = MSELossNp(a, y)

print("z:", z)
print("a:", a)
print("y:", y)
print("loss:", loss)

z: [-0.9640687  -0.12309539 -1.97287938 -1.13190608]
a: [0.27606431 0.46926495 0.12207995 0.24380951]
y: [0 1 1 0]
loss: 0.2970194729241582


In [37]:
def backprop(X, y, a):
    dL_da = 2 * (a - y) # (4,)
    da_dz = a * (1 - a) # (4,)
    dz_dW = X # (4, 2)
    dL_dz = dL_da * da_dz # (4,)

    num_samples = len(X)
    dL_dW = np.dot(dz_dW.T, dL_dz) / num_samples # (2, 4) @ (4,) -> (2,)

    dL_dB = np.mean(dL_dz) # average over all samples
    return dL_dW, dL_dB

In [38]:
# Train the neuron
epochs = 5000
lr = 0.01
for epoch in range(epochs):
    # forward
    z, a = forwardNp(X, W, bias)

    # loss
    loss = MSELossNp(a, y)

    # backward
    gradW, gradB = backprop(X, y, a)

    # update
    W = W - lr * gradW
    bias = bias - lr * gradB

    # print loss every x epochs
    if epoch % 500 == 0:
        print(f"loss at epoch {epoch}: {loss}")

loss at epoch 0: 0.2970194729241582
loss at epoch 500: 0.27822754561723373
loss at epoch 1000: 0.27269859517971623
loss at epoch 1500: 0.26998305881252466
loss at epoch 2000: 0.26768251725683295
loss at epoch 2500: 0.26539437301288393
loss at epoch 3000: 0.2631181805943811
loss at epoch 3500: 0.26093360960853973
loss at epoch 4000: 0.2589237130034216
loss at epoch 4500: 0.2571485382076084


In [39]:
# Test the neuron
_, a = forwardNp([0,0], W, bias)
print(a)
_, a = forwardNp([0,1], W, bias)
print(a)
_, a = forwardNp([1,0], W, bias)
print(a)
_, a = forwardNp([1,1], W, bias)
print(a)

0.45031486255443814
0.5811052830692718
0.37686788402184146
0.5059614834852955


---

In [40]:
# Model params - Two neurons
num_features = X.shape[1]
W1 = np.random.randn(num_features)
b1 = np.random.randn()
W2 = np.random.randn()
b2 = np.random.randn()

print(f"W1: {W1}\nW2: {W2}")
print(f"b1: {b1}\nb2: {b2}")

W1: [-0.61716596 -2.36883468]
W2: -0.47494997743722867
b1: -0.05390439970890896
b2: -0.1905429559679652


In [41]:
# Forward pass
def forward_two_neurons(W1, b1, W2, b2, X):
    _, a1 = forwardNp(X, W1, b1)
    _, a2 = forwardNp(a1, W2, b2)
    return a1, a2

In [42]:
# Test forward pass
a1, a2 = forward_two_neurons(W1, b1, W2, b2, X)
print(f"a1: {a1}\na2: {a2}")

a1: [0.48652716 0.08145508 0.33825721 0.04565531]
a2: [0.3961294  0.44294212 0.41309189 0.44714151]


In [43]:
# Backward pass
def backward_two_neurons(x, y, a1, a2, w2):
    num_samples = x.shape[0]   # () -> 4

    dl_da2 = 2 * (a2 - y)       # (4,)
    da2_dz2 = a2 * (1 - a2)     # (4, )
    dz2_dw2 = a1                # (4,)
    dz2_db2 = 1                 # ()

    dz2_da1 = w2                # ()
    da1_dz1 = a1 * (1 - a1)     # (4,)
    dz1_dw1 = x                 # (4, 2)
    dz1_db1 = 1                 # ()

    dl_dz2 = dl_da2 * da2_dz2   # (4,)

    dl_dw2 = np.dot(dl_dz2, dz2_dw2) / num_samples  # () which matches W2's shape
    dl_db2 = np.mean(dl_dz2 * dz2_db2)              # () which matches b2's shape

    dl_dz1 = dl_dz2 * dz2_da1 * da1_dz1                 # (4,) -> the gradient sent back to neuron 1

    dl_dw1 = np.dot(dz1_dw1.T, dl_dz1) / num_samples    # (2, 4) @ (4,) -> (2,) which matches W1's shape
    dl_db1 = np.mean(dl_dz1 * dz1_db1)                  # () which matches b1's shape

    return dl_dw2, dl_db2, dl_dw1, dl_db1

In [44]:
# Test backward pass
backward_two_neurons(X, y, a1, a2, W2)

(np.float64(-0.004089476853835365),
 np.float64(-0.037225243077193534),
 array([0.0064201, 0.0012985]),
 np.float64(0.0032407102494456733))

In [45]:
# Train 2 neurons
epochs = 5000
lr = 0.01
for epoch in range(epochs):
    # forward
    a1, a2 = forward_two_neurons(W1, b1, W2, b2, X)

    # loss
    loss = MSELossNp(a2, y)

    # backward
    dl_dw2, dl_db2, dl_dw1, dl_db1 = backward_two_neurons(X, y, a1, a2, W2)

    # update weights
    W1 = W1 - lr * dl_dw1
    b1 = b1 - lr * dl_db1

    W2 = W2 - lr * dl_dw2
    b2 = b2 - lr * dl_db2

    # print loss every x epochs
    if epoch % 500 == 0:
        print(f"loss at epoch {epoch}: {loss}")

loss at epoch 0: 0.25290716144919606
loss at epoch 500: 0.2486442743774791
loss at epoch 1000: 0.2472640939084093
loss at epoch 1500: 0.24665856106600959
loss at epoch 2000: 0.2462373622920474
loss at epoch 2500: 0.24584186138107272
loss at epoch 3000: 0.24542888494740456
loss at epoch 3500: 0.2449857060291857
loss at epoch 4000: 0.2445076678742672
loss at epoch 4500: 0.24399234199192887


In [46]:
# Test the 2 neurons
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [0,0])
print(a2)
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [0,1])
print(a2)
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [1,0])
print(a2)
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [1,1])
print(a2)

0.4512605767117757
0.5230803835008335
0.4892769726588258
0.530872531504698


---

In [47]:
# Model params - MLP
num_features = X.shape[1]
neurons_in_hidden_layer = 2
W1 = np.random.randn(neurons_in_hidden_layer, num_features)
b1 = np.random.randn(neurons_in_hidden_layer)
W2 = np.random.randn(neurons_in_hidden_layer)
b2 = np.random.randn()

print(f"W1: {W1}\nW2: {W2}")
print(f"b1: {b1}\nb2: {b2}")

W1: [[-0.93278698 -0.23326993]
 [-1.5674091  -0.60283871]]
W2: [ 0.99981108 -0.01809926]
b1: [ 0.01603954 -0.87972739]
b2: -0.7469956643031351


In [48]:
a1, a2 = forward_two_neurons(W1, b1, W2, b2, X) # works for MLP too
print(f"a1: {a1}\na2: {a2}")

a1: [[0.5040098  0.29323427]
 [0.17488855 0.18504014]
 [0.28562109 0.24731252]
 [0.07697636 0.15240829]]
a2: [0.43822021 0.35997118 0.385586   0.33787187]


In [49]:
# Backward pass - MLP
def backward_mlp(x, y, a1, a2, w2):
    num_samples = x.shape[0]   # () -> 4

    dl_da2 = 2 * (a2 - y)       # (4,)
    da2_dz2 = a2 * (1 - a2)     # (4, )
    dz2_dw2 = a1                # (4, 2)
    dz2_db2 = 1                 # ()

    dz2_da1 = w2                # (2,)
    da1_dz1 = a1 * (1 - a1)     # (4, 2)
    dz1_dw1 = x                 # (4, 2)
    dz1_db1 = 1                 # ()

    dl_dz2 = dl_da2 * da2_dz2   # (4,)

    dl_dw2 = np.dot(dl_dz2, dz2_dw2) / num_samples      # (2,) which matches W2's shape
    dl_db2 = np.mean(dl_dz2 * dz2_db2)                  # () which matches b2's shape

    dl_dz1 = (dl_dz2[:, None] * dz2_da1) * da1_dz1      # (4, 1) outer prod (2,) * (4, 2) -> (4, 2) the gradient sent back to layer 1
    # TODO: try the dot product version

    dl_dw1 = np.dot(dz1_dw1.T, dl_dz1) / num_samples    # (2, 4) @ (4, 2) -> (2, 2) which matches W1's shape
    dl_db1 = np.mean(dl_dz1 * dz1_db1, axis=0)          # (2,) which matches b1's shape

    return dl_dw2, dl_db2, dl_dw1, dl_db1

In [50]:
backward_mlp(X, y, a1, a2, W2)

(array([-0.00358577, -0.01006479]),
 np.float64(-0.05477432181816457),
 array([[-0.01216264,  0.00015684],
        [-0.00795248,  0.00011287]]),
 array([-0.00931798,  0.00015574]))

In [51]:
# Train MLP
epochs = 100000
lr = 0.5
for epoch in range(epochs):
    # forward
    a1, a2 = forward_two_neurons(W1, b1, W2, b2, X)

    # loss
    loss = MSELossNp(a2, y)

    # backward
    dl_dw2, dl_db2, dl_dw1, dl_db1 = backward_mlp(X, y, a1, a2, W2)

    # update weights
    W1 = W1 - lr * dl_dw1
    b1 = b1 - lr * dl_db1

    W2 = W2 - lr * dl_dw2
    b2 = b2 - lr * dl_db2

    # print loss every x epochs
    if epoch % 10000 == 0:
        print(f"loss at epoch {epoch}: {loss}")

loss at epoch 0: 0.27333395214493544
loss at epoch 10000: 0.0007620970169270559
loss at epoch 20000: 0.00031036608161199425
loss at epoch 30000: 0.00019271713687396796
loss at epoch 40000: 0.0001391893082298259
loss at epoch 50000: 0.0001087167852155085
loss at epoch 60000: 8.908666604379074e-05
loss at epoch 70000: 7.540433663263038e-05
loss at epoch 80000: 6.533106755484554e-05
loss at epoch 90000: 5.7610045903020194e-05


In [52]:
# Test MLP
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [0,0])
print(a2)
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [0,1])
print(a2)
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [1,0])
print(a2)
a1, a2 = forward_two_neurons(W1, b1, W2, b2, [1,1])
print(a2)

0.0065883243052754226
0.9931873882415434
0.9931778294640619
0.008346579210588909
